# 04 · Representaciones y catálogo de features (Etapa B)

**Pregunta.** ¿Qué representación maximiza macro-F1 bajo el protocolo agrupado por clústeres?

**Diseño experimental.** Modelo fijo (LinearSVC C=1) para aislar el efecto de la representación;
validación `StratifiedGroupKFold` de 5 folds agrupada por clúster de near-duplicados;
vectorizadores ajustados **solo** con el train de cada fold. Cada configuración tiene hipótesis
declarada antes de ejecutarse (ver `experiments.csv`).

Este notebook **no re-ejecuta** los experimentos: documenta y analiza sus resultados guardados en
`experiments/representaciones_resultados.json`.

## 0. Configuración

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src import config as cfg, feature_catalog
from src.config import set_seed

set_seed()
pd.set_option("display.width", 175)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 60)

representaciones = json.loads((cfg.EXPERIMENTS / "representaciones_resultados.json").read_text())
baselines = json.loads((cfg.EXPERIMENTS / "baselines_resultados.json").read_text())
E001 = json.loads((cfg.EXPERIMENTS / "E-001_resultados.json").read_text())
print("resultados cargados:", len(representaciones), "representaciones |", len(baselines), "baselines")

resultados cargados: 14 representaciones | 3 baselines


## 1. Matriz de representaciones

In [2]:
matriz = pd.DataFrame([
    {"id": key, "representacion": value["nombre"], "texto": value["texto"],
     "macro_f1": round(value["macro_f1_mean"], 4), "std": round(value["macro_f1_std"], 4),
     "weighted": round(value["weighted_f1_mean"], 4),
     "accuracy": round(value["accuracy_mean"], 4), "features": value["n_features"]}
    for key, value in representaciones.items()
]).sort_values("macro_f1", ascending=False).reset_index(drop=True)
matriz.to_csv(cfg.TABLES / "representaciones_comparativa.csv", index=False)
matriz

,id,representacion,texto,macro_f1,std,weighted,accuracy,features
0,E-008,char_wb 3-6 + canal de menciones,crudo,0.5557,0.0052,0.5548,0.5553,314420
1,E-003c,char_wb 3-6,crudo,0.5413,0.0049,0.5404,0.5418,300000
2,E-003b,char_wb 3-5,crudo,0.5408,0.0050,0.5399,0.5415,224628
3,E-004b,char_wb 3-6 + word 1-gram,crudo,0.5354,0.0037,0.5345,0.5354,339409
4,E-007a,union sin IDF (TF puro),crudo,0.5339,0.0046,0.5330,0.5341,336493
5,E-005c,solo quitar URLs + union,sin_urls,0.5277,0.0038,0.5267,0.5259,292170
6,E-005e,solo quitar puntuación + union,sin_punct,0.5264,0.0043,0.5254,0.5261,292313
7,E-003a,char_wb 2-4,crudo,0.5261,0.0048,0.5253,0.5273,121285
8,E-002a,word 1-gram,crudo,0.5025,0.0040,0.5015,0.5021,39409
9,E-002b,word 1-2 gram,crudo,0.4971,0.0034,0.4961,0.4973,111865


**OBSERVACIÓN.** Los n-gramas de carácter dominan: `char_wb 3-6` (0,5413) y `char_wb 3-5`
(0,5408) superan a todos los esquemas de palabra (`word 1-gram` 0,5025; `word 1-2` 0,4971;
`word 1-3` 0,4941). La unión palabra+carácter (E-001, 0,5344) queda **por debajo** del char solo.

**INTERPRETACIÓN.** Coincide con la literatura (Sapkota et al. 2015; PAN 2019, D-008): en texto
informal, corto y transliterado a ASCII, el n-grama de carácter captura morfología, abreviaturas
y ortografía, y no pierde información por tokenización. Añadir word n-grams diluye el modelo
lineal sin aportar señal nueva.

**CONSECUENCIA.** La representación base pasa a ser **char n-grams**, no la unión.

## 2. Descomposición de la normalización: el hallazgo de las menciones

In [3]:
descomposicion = matriz[matriz["id"].isin(["E-001", "E-005a", "E-005b", "E-005c", "E-005d", "E-005e"])]
descomposicion = descomposicion.sort_values("macro_f1", ascending=False)
referencia = float(E001["macro_f1_mean"])
descomposicion = descomposicion.assign(delta_vs_crudo=(descomposicion["macro_f1"] - referencia).round(4))
descomposicion[["id", "representacion", "texto", "macro_f1", "delta_vs_crudo", "features"]]

,id,representacion,texto,macro_f1,delta_vs_crudo,features
5,E-005c,solo quitar URLs + union,sin_urls,0.5277,-0.0067,292170
6,E-005e,solo quitar puntuación + union,sin_punct,0.5264,-0.0080,292313
11,E-005b,marcadores <url>/<user> + union,marcadores,0.4107,-0.1237,218709
12,E-005d,solo quitar menciones + union,sin_menciones,0.4094,-0.1250,260025
13,E-005a,union word 1-2 + char 3-5 (normalizado),normalizado,0.3902,-0.1442,183263


**OBSERVACIÓN (cuantitativa).** Partiendo del texto crudo (0,5344):
- quitar **solo las menciones** → **0,4094** (−0,1251, 0/5 folds a favor);
- sustituir menciones por el marcador `<user>` → **0,4107** (−0,1237);
- quitar solo URLs → 0,5277 (−0,0067);
- quitar solo puntuación → 0,5264 (−0,0080);
- normalización completa (URLs+menciones+RT+puntuación+minúsculas) → 0,3902 (−0,1442).

Además, tras normalizar, **7.240 tweets quedan vacíos** (frente a 104 originales) y 13.137 se
quedan con ≤1 token: hay muchísimos tweets que son *solo URL*, *solo menciones* o *solo hashtags*
(p. ej. `#TFZ @M_MartinezRed @RaisAlGhousia …` → `tfz`).

**INTERPRETACIÓN.** La señal no está solo en lo que se dice, sino en **a quién se menciona** y en
la **forma** del tweet. La identidad de las cuentas mencionadas (fandoms, medios, políticos)
revela el mundo social del autor, que es informativo de la edad. Reemplazar por `<user>` destruye
esa identidad, por eso también cae.

**CONSECUENCIA.** (1) el texto se mantiene **crudo**; (2) se crea un **canal dedicado** de
TF-IDF sobre los tokens de mención (experimento E-008).

## 3. Canal de menciones: confirmación de la hipótesis

In [4]:
e008 = representaciones["E-008"]
e003c = representaciones["E-003c"]
print(f"E-003c (char 3-6)              : {e003c['macro_f1_mean']:.4f} ± {e003c['macro_f1_std']:.4f}")
print(f"E-008  (char 3-6 + menciones)  : {e008['macro_f1_mean']:.4f} ± {e008['macro_f1_std']:.4f}")
print(f"mejora                         : {e008['macro_f1_mean'] - e003c['macro_f1_mean']:+.4f}")
print(f"features                       : {e003c['n_features']:,} -> {e008['n_features']:,}")
print()
print("F1 por clase (E-003c -> E-008):")
comparativa = pd.DataFrame({"char_3_6": e003c["per_class_f1"], "char_3_6_+_menciones": e008["per_class_f1"]})
comparativa["delta"] = (comparativa["char_3_6_+_menciones"] - comparativa["char_3_6"]).round(4)
comparativa.round(4)

E-003c (char 3-6)              : 0.5413 ± 0.0049
E-008  (char 3-6 + menciones)  : 0.5557 ± 0.0052
mejora                         : +0.0144
features                       : 300,000 -> 314,420

F1 por clase (E-003c -> E-008):


,char_3_6,char_3_6_+_menciones,delta
13-17,0.6155,0.6240,0.0085
18-24,0.4428,0.4539,0.0110
25-34,0.4291,0.4423,0.0133
35-49,0.4958,0.5172,0.0214
50-64,0.5974,0.6125,0.0151
65-xx,0.6673,0.6845,0.0172


## 4. Comparación estadística pareada entre folds

In [5]:
from scipy import stats

def fold_scores(experiment_id):
    return np.array([fold["macro_f1"] for fold in representaciones[experiment_id]["per_fold"]])

e001_folds = np.array([fold["macro_f1"] for fold in E001["per_fold"]])
comparaciones = [
    ("E-008", "E-003c", "canal de menciones vs char 3-6", fold_scores("E-008"), fold_scores("E-003c")),
    ("E-003c", "E-002b", "char 3-6 vs word 1-2", fold_scores("E-003c"), fold_scores("E-002b")),
    ("E-003c", "E-001", "char 3-6 vs union word+char", fold_scores("E-003c"), e001_folds),
    ("E-005d", "E-001", "quitar menciones vs crudo", fold_scores("E-005d"), e001_folds),
    ("E-003c", "E-003b", "char 3-6 vs char 3-5", fold_scores("E-003c"), fold_scores("E-003b")),
]
rows = []
for left, right, label, scores_left, scores_right in comparaciones:
    difference = scores_left - scores_right
    rows.append({"comparacion": label, "dif_media": round(float(difference.mean()), 4),
                 "folds_gana": f"{int((difference > 0).sum())}/5",
                 "p_wilcoxon": round(float(stats.wilcoxon(scores_left, scores_right).pvalue), 4)})
paired = pd.DataFrame(rows)
paired.to_csv(cfg.TABLES / "representaciones_pareado.csv", index=False)
print("Nota: con 5 folds el p mínimo alcanzable por Wilcoxon es 0,0625; la evidencia es la "
      "consistencia (5/5) más el tamaño de efecto.")
paired

Nota: con 5 folds el p mínimo alcanzable por Wilcoxon es 0,0625; la evidencia es la consistencia (5/5) más el tamaño de efecto.


,comparacion,dif_media,folds_gana,p_wilcoxon
0,canal de menciones vs char 3-6,0.0144,5/5,0.0625
1,char 3-6 vs word 1-2,0.0442,5/5,0.0625
2,char 3-6 vs union word+char,0.0069,5/5,0.0625
3,quitar menciones vs crudo,-0.1251,0/5,0.0625
4,char 3-6 vs char 3-5,0.0005,3/5,0.4375


## 5. Decisión de representación (D-010)

**Representación adoptada:** `TF-IDF char_wb(3,6) sobre texto crudo` ∪ `TF-IDF word(1,1) sobre los
tokens de mención` + `LinearSVC(C=1)`.

Evidencia: mejor macro-F1 de la matriz (0,5557 ± 0,0052), mejora consistente en 5/5 folds sobre
la alternativa char sola (+0,0144) y sobre la baseline E-001 (+0,0213), con costo de features
moderado (314k) y sin ningún preprocesamiento que destruya señal.

## 6. Catálogo de features

In [6]:
catalogo = feature_catalog.build_catalog()
catalogo.to_csv(cfg.REPORTS / "feature_catalog.csv", index=False)
print(f"catálogo: {len(catalogo)} filas | grupos: {catalogo['group'].nunique()}")
print(catalogo.groupby("group").size().to_string())
catalogo[["name", "group", "description", "hypothesis", "leakage_risk"]].head(12)

catálogo: 44 filas | grupos: 8
group
A - contenido (word n-grams)      1
B - caracteres (char n-grams)     1
C - estilo                       20
D - textometría                  15
E - POS/sintaxis                  1
F - sentimiento                   1
G - emoción/emoji                 3
H - léxico específico             2


,name,group,description,hypothesis,leakage_risk
0,word_tfidf_1_2,A - contenido (word n-grams),TF-IDF sublinear de unigramas y bigramas de palabra (min...,H7 (léxico temático),bajo: vectorizador ajustado en fold
1,char_wb_tfidf_3_6,B - caracteres (char n-grams),TF-IDF sublinear de n-gramas de carácter 3-6 con fronter...,H6 (ortografía/morfología),bajo: ajustado en fold
2,n_chars,D - textometría,Número de caracteres del tweet sin padding,H1,"nulo: local al documento, no aprende del corpus"
3,n_words,D - textometría,Número de palabras separadas por espacios,H1,"nulo: local al documento, no aprende del corpus"
4,n_unique_words,D - textometría,Tipos léxicos distintos en el tweet,H1,"nulo: local al documento, no aprende del corpus"
5,ttr,D - textometría,Type-Token Ratio,H1,"nulo: local al documento, no aprende del corpus"
6,cttr,D - textometría,TTR corregido por longitud,H1,"nulo: local al documento, no aprende del corpus"
7,hapax_ratio,D - textometría,Proporción de palabras que aparecen una sola vez,H1,"nulo: local al documento, no aprende del corpus"
8,lexical_entropy,D - textometría,Entropía de Shannon del tweet,H1,"nulo: local al documento, no aprende del corpus"
9,avg_word_len,D - textometría,Longitud media de palabra,H1,"nulo: local al documento, no aprende del corpus"


### Conclusiones de la Etapa B

| # | Observación | Interpretación | Consecuencia |
|---|---|---|---|
| B1 | char n-grams > word n-grams (+0,044) | el carácter captura morfología y ortografía informal | representación base = char_wb 3-6 |
| B2 | La unión word+char diluye (−0,007 vs char solo) | sin señal léxica nueva que aportar | se descarta la unión clásica |
| B3 | Quitar menciones cuesta −0,125 | la identidad de las cuentas mencionadas es señal fuerte | canal dedicado de menciones (+0,014) |
| B4 | Quitar URLs/puntuación cuesta −0,007/−0,008 | la forma del tweet es señal | texto crudo, sin normalizar |
| B5 | 7.240 tweets vacíos tras normalizar | muchos tweets son solo enlaces/menciones | la normalización agresiva es inviable aquí |
| B6 | TF puro ≈ TF-IDF (−0,0005) | el IDF aporta poco con clases balanceadas | se conserva IDF sublinear (sin costo) |

**Siguiente:** Fase 13-19 — modelos clásicos sobre la representación ganadora, análisis de errores,
ajuste de macro-F1 (umbral/prior por clase) y ensambles.